### SPO Data Collection: Scraper & Extractor

This pipeline consists of two components that work in sequence:

---

#### Component 1: `SPOExtractor` — PDF Downloader

Fetches the Hof University Study and Examination Regulations (SPO) page and
downloads English-version PDFs for all degree programs.

- **Source URL:** `https://www.hof-university.com/studying-at-hof-university/services-and-support/student-affairs/study-and-examination-regulations.html`
- **Tool:** `requests` + `BeautifulSoup` (static HTML)
- **Output:** `spo_pdfs/<Program Name>.pdf`

##### Scraping Pipeline

```
1. Fetch SPO page HTML
2. Find all accordion items (one per degree program)
3. For each accordion:
   a. Extract program name from accordion header
   b. Collect all PDF links from accordion body
   c. Classify each link as English or German:
      - Check link text for "english" / "englisch" / "en)"
      - Check parent element text
      - Check URL pattern (_EN., /EN/)
4. Download first (most recent) English PDF per program
5. Save as spo_pdfs/<sanitized_program_name>.pdf
```

---


In [2]:
import requests
from bs4 import BeautifulSoup
import re
import os
from urllib.parse import urljoin
import time

class SPOExtractor:
    def __init__(self):
        self.base_url = "https://www.hof-university.com"
        self.spo_url = "https://www.hof-university.com/studying-at-hof-university/services-and-support/student-affairs/study-and-examination-regulations.html"
        self.output_dir = "debug_data/spo_pdfs"
        
    def fetch_page(self):
        """Fetch the SPO page content"""
        print(f"Fetching page: {self.spo_url}")
        try:
            response = requests.get(self.spo_url, timeout=30)
            response.raise_for_status()
            print("Page fetched successfully")
            return response.text
        except Exception as e:
            print(f"Error fetching page: {e}")
            return None
    
    def parse_spo_programs(self, html_content):
        """Parse HTML and extract program SPO information"""
        soup = BeautifulSoup(html_content, 'html.parser')
        
        programs = []
        
        # Find all accordion items
        accordion_items = soup.find_all('div', class_='accordion-item')
        print(f"\nFound {len(accordion_items)} accordion items")
        
        for item in accordion_items:
            # Extract program name from accordion header
            header = item.find('h4', class_='accordion-header')
            if not header:
                continue
            
            button = header.find('button', class_='accordion-button')
            if not button:
                continue
            
            # Get program name (remove the span element text)
            program_name = button.get_text(strip=True)
            
            # Find all PDF links in this accordion
            accordion_body = item.find('div', class_='accordion-body')
            if not accordion_body:
                continue
            
            pdf_links = accordion_body.find_all('a', href=re.compile(r'\.pdf$', re.IGNORECASE))
            
            if not pdf_links:
                continue
            
            # Extract English PDFs only
            english_pdfs = []
            german_pdfs = []
            
            for link in pdf_links:
                link_text = link.get_text(strip=True)
                pdf_url = link.get('href')
                
                # Make absolute URL
                if pdf_url.startswith('/'):
                    pdf_url = urljoin(self.base_url, pdf_url)
                
                # Check if English version (handle multiple variations)
                is_english = False
                
                # Check link text for English indicators
                link_lower = link_text.lower()
                if any(word in link_lower for word in ['english', 'englisch', 'en)']):
                    is_english = True
                
                # IMPORTANT: Also check the parent <td> or <p> element text
                # Sometimes "(English version)" is outside the <a> tag
                parent = link.parent
                if parent and parent.name in ['td', 'p', 'div']:
                    parent_text = parent.get_text(strip=True).lower()
                    if any(word in parent_text for word in ['english', 'englisch']):
                        # Make sure it's not the German version
                        if 'german' not in parent_text and 'deutsch' not in parent_text:
                            is_english = True
                
                # Also check if URL contains _EN or EN_ patterns
                url_upper = pdf_url.upper()
                if '_EN.' in url_upper or '_EN_' in url_upper or '/EN/' in url_upper:
                    is_english = True
                
                if is_english:
                    english_pdfs.append({
                        'url': pdf_url,
                        'description': link_text,
                        'filename': os.path.basename(pdf_url)
                    })
                else:
                    german_pdfs.append({
                        'url': pdf_url,
                        'description': link_text,
                        'filename': os.path.basename(pdf_url)
                    })
            
            if english_pdfs or german_pdfs:
                programs.append({
                    'program_name': program_name,
                    'english_pdfs': english_pdfs,
                    'german_pdfs': german_pdfs
                })
        
        return programs
    
    def sanitize_filename(self, name):
        """Convert program name to valid filename - keep exact name, only remove truly invalid chars"""
        # Only remove characters that are actually invalid in filenames
        # Windows: < > : " / \ | ? *
        invalid_chars = '<>:"/\\|?*'
        for char in invalid_chars:
            name = name.replace(char, '')
        return name.strip()
    
    def download_pdf(self, pdf_url, save_path):
        """Download a single PDF file"""
        try:
            print(f"  Downloading: {pdf_url}")
            response = requests.get(pdf_url, timeout=30)
            response.raise_for_status()
            
            with open(save_path, 'wb') as f:
                f.write(response.content)
            
            print(f"  Saved to: {save_path}")
            return True
        except Exception as e:
            print(f" Error downloading: {e}")
            return False
    
    def extract_first_english_pdf(self):
        """Extract the first English PDF only"""
        # Fetch page
        html_content = self.fetch_page()
        if not html_content:
            return None
        
        # Parse programs
        programs = self.parse_spo_programs(html_content)
        
        if not programs:
            print("\n No programs found")
            return None
        
        print(f"\nFound {len(programs)} programs with SPO documents")
        
        # Find first program with English PDF
        for program in programs:
            if program['english_pdfs']:
                print(f"\n Program: {program['program_name']}")
                print(f"   English PDFs available: {len(program['english_pdfs'])}")
                
                # Get first English PDF
                first_pdf = program['english_pdfs'][0]
                print(f"\n   Selected PDF: {first_pdf['description']}")
                print(f"   URL: {first_pdf['url']}")
                
                # Create output directory
                os.makedirs(self.output_dir, exist_ok=True)
                
                # Create filename - ONLY program name
                sanitized_name = self.sanitize_filename(program['program_name'])
                new_filename = f"{sanitized_name}.pdf"
                save_path = os.path.join(self.output_dir, new_filename)
                
                print(f"   Saving as: {new_filename}")
                
                # Download
                success = self.download_pdf(first_pdf['url'], save_path)
                
                if success:
                    return {
                        'program_name': program['program_name'],
                        'pdf_info': first_pdf,
                        'local_path': save_path
                    }
                else:
                    return None
        
        print("\nNo English PDFs found")
        return None
    
    def download_all_english_spos(self):
        """Download all English SPO PDFs for all programs"""
        # Fetch page
        html_content = self.fetch_page()
        if not html_content:
            return None
        
        # Parse programs
        programs = self.parse_spo_programs(html_content)
        
        print(f"DOWNLOADING ALL ENGLISH SPO PDFs")
        print(f"Total programs found: {len(programs)}\n")
        
        # Create output directory
        os.makedirs(self.output_dir, exist_ok=True)
        
        downloaded_count = 0
        failed_count = 0
        results = []
        
        for i, program in enumerate(programs, 1):
            program_name = program['program_name']
            english_pdfs = program['english_pdfs']
            
            if not english_pdfs:
                print(f"{i}. {program_name}")
                print(f"No English PDFs available (skipping)")
                continue
            
            print(f"\n{i}. {program_name}")
            print(f"English PDFs available: {len(english_pdfs)}")
            
            # Download the most recent English PDF (first one)
            first_pdf = english_pdfs[0]
            print(f"Downloading: {first_pdf['description']}")
            
            # Create filename from program name - ONLY program name, nothing else
            sanitized_name = self.sanitize_filename(program_name)
            new_filename = f"{sanitized_name}.pdf"
            save_path = os.path.join(self.output_dir, new_filename)
            
            # Download
            success = self.download_pdf(first_pdf['url'], save_path)
            
            if success:
                downloaded_count += 1
                results.append({
                    'program_name': program_name,
                    'pdf_info': first_pdf,
                    'local_path': save_path,
                    'status': 'success'
                })
                print(f"   Saved as: {new_filename}")
            else:
                failed_count += 1
                results.append({
                    'program_name': program_name,
                    'pdf_info': first_pdf,
                    'status': 'failed'
                })
                print(f"Failed to download")
            
            # Be nice to the server - add a small delay
            time.sleep(1)
        

        return results
    
    def extract_all_programs(self):
        """Extract all programs and their PDFs info (for analysis)"""
        # Fetch page
        html_content = self.fetch_page()
        if not html_content:
            return None
        
        # Parse programs
        programs = self.parse_spo_programs(html_content)
        
      
        print(f"COMPLETE SPO EXTRACTION - INFO ONLY")
        print(f"Total programs found: {len(programs)}\n")
        
        for i, program in enumerate(programs, 1):
            print(f"{i}. {program['program_name']}")
            print(f"English PDFs: {len(program['english_pdfs'])}")
            print(f"German PDFs: {len(program['german_pdfs'])}")
            
            if program['english_pdfs']:
                for pdf in program['english_pdfs']:
                    print(f"• {pdf['description']}")
        
        return programs


def main():
    """Main function - now downloads ALL English PDFs"""

    extractor = SPOExtractor()
    
    # Download all English SPO PDFs
    results = extractor.download_all_english_spos()
    
    if results:
       
        print("\nDOWNLOAD COMPLETE!")
        print(f"\nCheck the '{extractor.output_dir}/' folder for all downloaded PDFs")
        
       
    else:
        print("\nFailed to extract PDFs")


# Alternative function for testing - download only first one
def main_test_single():
    """Test function - downloads only the first English PDF"""
    print("SPO PDF EXTRACTOR - FIRST ENGLISH PDF (TEST)")
    
    extractor = SPOExtractor()
    
    # Extract first English PDF
    result = extractor.extract_first_english_pdf()
    
    if result:
        print(f"Program: {result['program_name']}")
        print(f"PDF: {result['pdf_info']['description']}")
        print(f"Saved to: {result['local_path']}")
    else:
        print("\nFailed to extract PDF")


if __name__ == "__main__":
    # Download all English SPO PDFs
    main()

Fetching page: https://www.hof-university.com/studying-at-hof-university/services-and-support/student-affairs/study-and-examination-regulations.html
Page fetched successfully

Found 14 accordion items
DOWNLOADING ALL ENGLISH SPO PDFs
Total programs found: 14


1. Computer Science (B.Sc.)
English PDFs available: 2
Downloading: SER CS from 01.10.2023 (English version)
  Downloading: https://www.hof-university.com/fileadmin/user_upload/studienbuero/spo-s/spo-in-englisch/SPO_CS_2020_EN_2023-08-03.pdf
  Saved to: debug_data/spo_pdfs/Computer Science (B.Sc.).pdf
   Saved as: Computer Science (B.Sc.).pdf

2. Innovative Textilien (B.Sc.)
English PDFs available: 2
Downloading: SER IN from 25.04.2024 (English version)
  Downloading: https://www.hof-university.com/fileadmin/user_upload/studienbuero/spo-s/spo-in-englisch/SPO-IN_2024-00_EN.pdf
  Saved to: debug_data/spo_pdfs/Innovative Textilien (B.Sc.).pdf
   Saved as: Innovative Textilien (B.Sc.).pdf

3. Applied Research in Computer Science (M.Sc


## Component 2: `SPOTableExtractor` — PDF Parser

Parses the downloaded SPO PDFs to extract structured content: filtered
regulatory sections and module tables.

- **Input:** `spo_pdfs/*.pdf`
- **Tool:** `pdfplumber`
- **Output:** `debug_data/spo_clean_data.json`

##### Extraction Pipeline

```
For each PDF:
  1. Extract full text with pdfplumber
  2. Parse §-numbered sections → keep only relevant ones
  3. Extract module tables from all pages
  4. Return { program_name, sections, modules }
```

##### Section Filtering (`should_keep_section`)

Only §-sections matching these topics are retained:

| Kept Sections |
|---------------|
| Admission Requirements (all variants) |
| Internship / Internship Report |
| Master's Thesis / Final Theses |
| Compulsory Elective Modules |
| Practical Semester with Bachelor Thesis |
| Third Re-Examinations |

##### Extracted Module Fields

| Field | Description |
|-------|-------------|
| `module_number` | Numeric, decimal, or alphanumeric module ID |
| `module_name` | Cleaned module title |
| `module_group` | Section grouping (e.g. Core Module, Elective) |
| `credits` | ECTS credit points (3–30) |
| `sws` | Contact hours per week (2–8) |
| `course_type` | Format code (e.g. `SU`, `Ü`, `Pr`, `SU,Ü`) |
| `examination_type` | Exam format code (e.g. `schrP90`, `StA`, `mdlP`) |



In [4]:
import os
import re
import json
import pdfplumber
from typing import List, Dict, Any
from pathlib import Path


class SPOTableExtractor:
    def __init__(self, pdf_directory="debug_data/spo_pdfs", debug_programs=None):
        self.pdf_directory = pdf_directory
        self.debug_programs = debug_programs or []
        
        # Define section titles to keep
        self.sections_to_keep = {
            'internship',
            'master thesis',
            'final theses',
            'internship report',
            'compulsory elective modules',
            "master's thesis",
            'practical semester with practical research project and bachelor thesis',
            "internships and master's thesis",
            'specific admission requirements',
            'admission requirements für the master\'s program',
            'admission requirements for modules',
            'third re-examinations',
            'admission requirements for the master\'s degree program',
            'admission requirements',
            'specific admission requirement',
            'admission requirements for individual modules'
        }
    
    def should_keep_section(self, section_title: str) -> bool:
        """Check if section title matches any of the titles to keep"""
        title_normalized = section_title.replace("\u2019", "'").replace("\u2018", "'").replace("`", "'")
        title_lower = title_normalized.lower().strip()
        
        if title_lower in self.sections_to_keep:
            return True
        
        if 'master thesis' in title_lower or "master's thesis" in title_lower or 'masters thesis' in title_lower:
            return True
        
        if title_lower == 'internship' or title_lower == 'internships':
            return True
        
        if 'internship report' in title_lower:
            return True
            
        if 'internships and master' in title_lower:
            return True
            
        if 'internship and master' in title_lower:
            return True
            
        if 'practical semester with practical research project and bachelor thesis' in title_lower:
            return True
        
        if 'compulsory elective modules' in title_lower:
            return True
            
        if 'final theses' in title_lower or 'final thesis' in title_lower:
            return True
        
        admission_patterns = ['admission requirement', 'third re-examination']
        
        for pattern in admission_patterns:
            if pattern in title_lower:
                return True
        
        return False
    
    def extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract text from PDF"""
        try:
            text = ""
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
            return text
        except Exception as e:
            print(f"    Error: {e}")
            return ""
    
    def split_title_and_content(self, section_title: str) -> tuple:
        """Split section title from content if they're on the same line"""
        section_title = section_title.replace("\u2019", "'").replace("\u2018", "'")
        
        sentence_pattern = r'^(.*?)\s+(\d+)\s*([A-Z].*)'
        
        match = re.match(sentence_pattern, section_title)
        if match:
            title_part = match.group(1).strip()
            sentence_num = match.group(2)
            content_part = f"{sentence_num}{match.group(3)}"
            
            if len(title_part) < 150:
                return (title_part, content_part)
        
        paren_pattern = r'^(.*?)\s+\((\d+)\)\s*(.*)'
        match = re.match(paren_pattern, section_title)
        if match:
            title_part = match.group(1).strip()
            if len(title_part) < 150:
                content_part = f"({match.group(2)}) {match.group(3)}"
                return (title_part, content_part)
        
        return (section_title, None)
    
    def parse_sections(self, text: str) -> Dict[str, str]:
        """Parse sections with filtering and return as key-value pairs"""
        sections_dict = {}
        lines = text.split('\n')
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            section_match = re.match(r'^§\s*(\d+)\s*(.*)', line)
            
            if section_match:
                section_number = section_match.group(1)
                first_title_part = section_match.group(2).strip()
                
                i += 1
                
                title_parts = [first_title_part] if first_title_part else []
                
                title_lines_collected = 0
                while i < len(lines) and title_lines_collected < 2:
                    next_line = lines[i].strip()
                    
                    if not next_line:
                        i += 1
                        continue
                    
                    if re.match(r'^[\(\[]?\d+[\)\]]', next_line):
                        break
                    
                    if re.match(r'^§', next_line):
                        break
                    
                    if re.match(r'^(Annex|Appendix)', next_line, re.IGNORECASE):
                        break
                    
                    title_parts.append(next_line)
                    i += 1
                    title_lines_collected += 1
                
                section_title = ' '.join(title_parts).strip()
                
                should_keep = self.should_keep_section(section_title)
                
                if not should_keep:
                    while i < len(lines):
                        next_line = lines[i].strip()
                        if re.match(r'^§', next_line) or re.match(r'^(Annex|Appendix)\s*\(', next_line, re.IGNORECASE):
                            break
                        i += 1
                    continue
                
                clean_title, initial_content = self.split_title_and_content(section_title)
                
                content_parts = []
                
                if initial_content:
                    content_parts.append(initial_content)
                
                while i < len(lines):
                    next_line = lines[i].strip()
                    
                    if re.match(r'^§', next_line) or re.match(r'^(Annex|Appendix)\s*\(', next_line, re.IGNORECASE):
                        break
                    
                    if next_line:
                        content_parts.append(next_line)
                    
                    i += 1
                
                content = '\n'.join(content_parts).strip()
                
                # Store as key-value pair: section_title -> content
                sections_dict[clean_title] = content
                
                continue
            
            i += 1
        
        return sections_dict
    
    def is_module_number(self, text: str) -> bool:
        """Check if text represents a valid module number"""
        if not text:
            return False
        
        text = str(text).strip()
        
        # Handle pure digits (including 4-digit module codes)
        if text.isdigit():
            num = int(text)
            # Allow 1-200 for traditional numbering AND 1000-9999 for code-based numbering
            if (1 <= num <= 200) or (1000 <= num <= 9999):
                return True
            return False
        
        # Handle decimal notation (1.1, 1.2, etc.)
        if re.match(r'^\d+\.\d+$', text):
            return True
        
        # Handle alphanumeric (1A, 2B, etc.)
        if re.match(r'^\d+[A-Za-z]$', text):
            return True
        
        # Handle Roman numerals
        if text in ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']:
            return True
        
        return False
    
    def clean_module_name(self, name: str) -> str:
        """Clean module name"""
        if not name:
            return ""
        
        name = str(name).strip()
        
        patterns_to_remove = [
            r'^Module\s+\d+[:\-\s]*',
            r'^\d+\.\s*',
            r'\s+\([^)]*credit[^)]*\)\s*$',
            r'\s+\([^)]*ECTS[^)]*\)\s*$',
        ]
        
        for pattern in patterns_to_remove:
            name = re.sub(pattern, '', name, flags=re.IGNORECASE)
        
        name = ' '.join(name.split())
        
        return name
    
    def extract_modules_from_tables(self, pdf_path: str) -> List[Dict[str, Any]]:
        """Extract modules from tables"""
        modules = []
        program_name = Path(pdf_path).stem
        debug_mode = program_name in self.debug_programs
        
        found_any_tables = False
        
        try:
            with pdfplumber.open(pdf_path) as pdf:
                current_group = None
                
                for page_num, page in enumerate(pdf.pages, 1):
                    tables = page.extract_tables()
                    
                    if not tables:
                        continue
                    
                    found_any_tables = True
                    
                    if debug_mode:
                        print(f"\nPage {page_num}: Found {len(tables)} table(s)")
                    
                    text = page.extract_text()
                    
                    if text:
                        text_lines = text.split('\n')
                        for line in text_lines:
                            line_clean = line.strip()
                            
                            if any(marker in line_clean.lower() for marker in [
                                'core module', 'compulsory module', 'pflichtmodul',
                                'elective module', 'wahlpflichtmodul', 'specialization'
                            ]):
                                current_group = line_clean
                                if debug_mode:
                                    print(f"  Found section: {current_group}")
                                break
                    
                    for table_num, table in enumerate(tables, 1):
                        if not table or len(table) < 2:
                            continue
                        
                        if debug_mode:
                            print(f"\n  Table {table_num}:")
                            print(f"    Rows: {len(table)}, Columns: {len(table[0]) if table else 0}")
                            # Print first few rows for debugging
                            for i, row in enumerate(table[:5]):
                                print(f"    Row {i}: {row}")
                        
                        # More flexible header detection
                        header_row = None
                        data_start_row = 0
                        
                        # Check first 5 rows for header-like content
                        for i, row in enumerate(table[:5]):
                            row_text = ' '.join([str(cell).lower() if cell else '' for cell in row])
                            
                            # Look for header keywords
                            if any(keyword in row_text for keyword in [
                                'module', 'modul', 'course', 'title', 'ects', 'credit', 'sws', 
                                'examination', 'prüfung', 'no.', 'nr.', 'form', 'lv', 'numbers'
                            ]):
                                header_row = i
                                data_start_row = i + 1
                                if debug_mode:
                                    print(f"    Found header at row {i}")
                                break
                        
                        # If no clear header found, assume data starts at row 0 or 1
                        if header_row is None:
                            first_row = table[0] if table else []
                            if first_row and any(cell and re.match(r'^\d+\.?\d*$', str(cell).strip()) for cell in first_row):
                                data_start_row = 0
                                if debug_mode:
                                    print(f"    No header found, starting from row 0")
                            else:
                                data_start_row = 1
                                if debug_mode:
                                    print(f"    No header found, starting from row 1")
                        
                        # Process data rows
                        for row_num, row in enumerate(table[data_start_row:], start=data_start_row):
                            if not row or all(not cell or str(cell).strip() == '' for cell in row):
                                continue
                            
                            # Skip rows that look like headers (but not section headers)
                            row_text_lower = ' '.join([str(cell).lower() if cell else '' for cell in row])
                            if any(keyword in row_text_lower for keyword in [
                                'module number', 'type of course', 'examination', 'credit points'
                            ]) and not any(section in row_text_lower for section in ['core', 'elective', 'compulsory']):
                                if debug_mode:
                                    print(f"      Skipping header row {row_num}")
                                continue
                            
                            # Filter out None values to get actual data columns
                            non_none_cells = [(i, cell) for i, cell in enumerate(row) if cell is not None and str(cell).strip() != '']
                            
                            if debug_mode and non_none_cells:
                                print(f"      Row {row_num} non-None cells: {non_none_cells}")
                            
                            if len(non_none_cells) < 2:
                                if debug_mode:
                                    print(f"      Skipping row {row_num} - insufficient data")
                                continue
                            
                            # Extract module info from non-None cells
                            module_number = None
                            module_name = None
                            name_idx = -1
                            name_cell_idx = -1  # Index in non_none_cells list
                            
                            # Look for module number in first non-None cells
                            for idx, (col_idx, cell) in enumerate(non_none_cells[:3]):
                                cell_str = str(cell).strip()
                                
                                if self.is_module_number(cell_str):
                                    # Special check: skip section headers (single digit with no decimal)
                                    if cell_str.isdigit() and int(cell_str) < 10 and len(non_none_cells) > idx + 1:
                                        next_cell = str(non_none_cells[idx + 1][1]).strip().lower()
                                        # Check if next cell looks like a section name
                                        if any(keyword in next_cell for keyword in 
                                            ['deutsch', 'german', 'kernmodule', 'core module', 'praktika', 'internship']):
                                            # Check if this is really a section header by looking ahead
                                            next_row_idx = row_num + 1
                                            if next_row_idx < len(table):
                                                next_row = table[next_row_idx]
                                                next_non_none = [c for c in next_row if c is not None and str(c).strip() != '']
                                                if next_non_none and next_non_none[0]:
                                                    next_first = str(next_non_none[0]).strip()
                                                    # If next row has X.Y format, this is a section header
                                                    if re.match(rf'^{cell_str}\.\d+$', next_first):
                                                        if debug_mode:
                                                            print(f"      Skipping section header: {cell_str} {next_cell}")
                                                        module_number = None
                                                        break
                                    
                                    module_number = cell_str
                                    
                                    # Look for module name in next non-None cell
                                    if idx + 1 < len(non_none_cells):
                                        candidate_name = str(non_none_cells[idx + 1][1]).strip()
                                        # Make sure it's not just a number or course type
                                        if (len(candidate_name) > 2 and 
                                            not candidate_name.isdigit() and
                                            not re.match(r'^[A-Z]{1,3}(,\s*[A-Z]{1,3})*$', candidate_name)):
                                            module_name = candidate_name
                                            name_idx = non_none_cells[idx + 1][0]  # Original column index
                                            name_cell_idx = idx + 1  # Index in non_none_cells
                                            break
                            
                            if not module_number or not module_name:
                                if debug_mode:
                                    print(f"      Skipping row {row_num} - no valid module")
                                continue
                            
                            # Store original name before cleaning
                            original_module_name = module_name
                            
                            # Clean module name
                            module_name = self.clean_module_name(module_name)
                            module_name = re.sub(r'\s*\(gemäß.*?\)', '', module_name)
                            module_name = ' '.join(module_name.split())
                            
                            if debug_mode:
                                print(f"        Found module: {module_number}. {module_name}")
                            
                            # Extract credits, SWS, course type, exam type from remaining non-None cells
                            credits = None
                            sws = None
                            course_type = None
                            exam_type = None
                            
                            # Get cells after the name (using the index in non_none_cells, not the cleaned name)
                            remaining_cells = non_none_cells[name_cell_idx + 1:] if name_cell_idx + 1 < len(non_none_cells) else []
                            
                            # Find credits (last numeric cell, typically 3-30)
                            for col_idx, cell in reversed(remaining_cells):
                                cell_str = str(cell).strip()
                                if cell_str.isdigit():
                                    num = int(cell_str)
                                    if 3 <= num <= 30:
                                        credits = num
                                        break
                            
                            # Find SWS (numeric cell 2-8, before credits)
                            for col_idx, cell in remaining_cells:
                                cell_str = str(cell).strip()
                                if cell_str.isdigit():
                                    num = int(cell_str)
                                    if 2 <= num <= 8 and num != credits:
                                        sws = num
                                        break
                            
                            # Build text from all remaining cells for pattern matching
                            row_as_text = ' '.join([str(cell) for _, cell in remaining_cells])
                            
                            # Extract course type
                            course_patterns = [
                                r'\b(SU\s*,\s*Ü)\b',
                                r'\b(SU\s*,\s*Pr)\b', 
                                r'\b(SU\s*,\s*V)\b',
                                r'\b(SU)\b',
                                r'\b(Ü)\b',
                                r'\b(Pr)\b',
                                r'\b(V)\b',
                                r'\b(S)\b',
                                r'\b(Internship)\b'
                            ]
                            
                            for pattern in course_patterns:
                                match = re.search(pattern, row_as_text, re.IGNORECASE)
                                if match:
                                    course_type = match.group(1).replace(' ', '')
                                    break
                            
                            # Extract exam type
                            exam_patterns = [
                                r'schrP\d+', r'mdlP\d*', r'Kl\d+', r'KI\d+', r'CP\d+',
                                r'Koll\d*', r'StA\s+mit\s+Präs', r'StA\d*', r'SA\b', r'PfP\b', 
                                r'PrB\b', r'AA\b', r'Ref\d*', r'Präs\d*', r'PrjA', r'BA\b', r'MA\b',
                                r'Internship report',
                                r'(?<![a-zäöüß])P[¹²³⁴⁵⁶⁷⁸⁹⁰\d]+',
                                r'(?<![a-zäöüß])P\b',
                            ]
                            
                            exam_matches = []
                            for pattern in exam_patterns:
                                matches = re.findall(pattern, row_as_text, re.IGNORECASE)
                                exam_matches.extend(matches)
                            
                            exam_matches = [e for e in exam_matches if e.strip() not in ['mit', 'mit ']]
                            
                            if exam_matches:
                                seen = set()
                                unique_exams = []
                                for exam in exam_matches:
                                    exam_lower = exam.lower()
                                    if exam_lower not in seen:
                                        seen.add(exam_lower)
                                        unique_exams.append(exam)
                                exam_type = ', '.join(unique_exams)
                            
                            # Check for regulation references
                            if not exam_type:
                                if re.search(r'according to|gemäß|module handbook', row_as_text, re.IGNORECASE):
                                    exam_type = "according to relevant SPO/module handbook"
                            
                            # Add module if we have required fields
                            if module_number and module_name and credits:
                                modules.append({
                                    'module_number': module_number,
                                    'module_name': module_name,
                                    'module_group': current_group,
                                    'credits': credits,
                                    'sws': sws,
                                    'course_type': course_type,
                                    'examination_type': exam_type
                                })
                                
                                if debug_mode:
                                    print(f"        Added: {module_number}. {module_name} ({credits} ECTS, SWS: {sws})")
                            else:
                                if debug_mode:
                                    print(f"        Missing required fields - Num: {module_number}, Name: {module_name}, Credits: {credits}")
        
        except Exception as e:
            print(f"    Table extraction error: {e}")
            import traceback
            traceback.print_exc()
        
        if found_any_tables and not modules:
            print(f"WARNING: Found tables but extracted 0 modules for {program_name}")
        
        return modules
    
    def parse_single_pdf(self, pdf_path: str) -> Dict[str, Any]:
        """Parse single PDF"""
        program_name = Path(pdf_path).stem
        
        print(f"\nParsing: {program_name}")
        
        text = self.extract_text_from_pdf(pdf_path)
        
        if not text:
            return None
        
        sections = self.parse_sections(text)
        print(f"    Found {len(sections)} relevant sections")
        
        modules = self.extract_modules_from_tables(pdf_path)
        print(f"    Extracted {len(modules)} modules")
        
        return {
            'program_name': program_name,
            'sections': sections,
            'modules': modules
        }
    
    def parse_all_pdfs(self) -> Dict[str, Any]:
        """Parse all PDFs"""
        pdf_files = list(Path(self.pdf_directory).glob("*.pdf"))
        
        print(f"Found {len(pdf_files)} PDF files")
        
        all_data = {}
        
        for pdf_file in pdf_files:
            parsed_data = self.parse_single_pdf(str(pdf_file))
            
            if parsed_data:
                program_name = parsed_data['program_name']
                all_data[program_name] = parsed_data
        
        return all_data
    
    def save_clean_data(self, data: Dict[str, Any], output_file: str = "debug_data/spo_data.json"):
        """Save clean structured data"""
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
     
        print(f"    Saved to: {output_file}")


def main():
    """Main function"""

    os.makedirs("data", exist_ok=True)
    os.makedirs("debug_data", exist_ok=True)

    extractor = SPOTableExtractor(pdf_directory="debug_data/spo_pdfs")
    
    all_data = extractor.parse_all_pdfs()
    
    if not all_data:
        print("\n   No data parsed")
        return
    
    extractor.save_clean_data(all_data, output_file="debug_data/spo_clean_data.json")
    
    print("\nRelevant sections and all modules have been extracted!")


if __name__ == "__main__":
    main()

Found 14 PDF files

Parsing: Operational Excellence (M.B.A. and Eng.)
    Found 3 relevant sections
    Extracted 20 modules

Parsing: Artificial Intelligence and Robotics (M.Sc.)
    Found 2 relevant sections
    Extracted 20 modules

Parsing: Innovative Textilien (B.Sc.)
    Found 4 relevant sections
    Extracted 37 modules

Parsing: Software Engineering for Industrial Applications (M.Eng.)
    Found 2 relevant sections
    Extracted 23 modules

Parsing: Applied Research in Computer Science (M.Sc.)
    Found 2 relevant sections
    Extracted 9 modules

Parsing: Digitalization and Innovation (M.B.A.)
    Found 3 relevant sections
    Extracted 20 modules

Parsing: Sustainability Management (M.B.A. and Eng.)
    Found 2 relevant sections
    Extracted 28 modules

Parsing: Cross Cultural Nursing Practice (M.Sc.)
    Found 3 relevant sections
    Extracted 14 modules

Parsing: General Management (M.B.A.)
    Found 3 relevant sections
    Extracted 20 modules

Parsing: Sustainable Water 